In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/Advertising.csv?raw=true')

In [ ]:
data

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9
...,...,...,...,...,...
195,196,38.2,3.7,13.8,7.6
196,197,94.2,4.9,8.1,9.7
197,198,177.0,9.3,6.4,12.8
198,199,283.6,42.0,66.2,25.5


## The Main Goals

We want to predict the Sales by using the feature values of TV, Radio and Newspaper advertising. We think of a weighted combination of TV, Radio and Newspaper amounts from commercials.

In [3]:
def MSE(w,x,y):
  return np.mean((y-x@w)**2) # here we use the "At" sign to represent the linear combiantion of columns in x by using the weights in w


In [6]:
# we use all the data as train data
x = data.loc[:,['TV','Radio','Newspaper']].values
y = data['Sales'].values

In [ ]:
# here the input features are a Numpy array
x

In [4]:
# say pick a random set of weights
w = [1,2,3]

In [7]:
MSE(w,x,y)

87288.4676

We want to apply gradient descent to minimize the MSE. This means we need to differentiate MSE as a function of the weights.
A good idea is to standardize x and include a bias term OR center the values of y.

In [ ]:
# you can center y but this may not be ideal in real situations
# an alternative would be to have a bias term w0 in the vector of the weights
y_centered = y - np.mean(y)

In [8]:
def zscore(x):
  return (x-np.mean(x))/np.std(x)

In [9]:
x_scaled = zscore(x)

In [10]:
# we need to include a column of 1s in x_scaled to accommodate the bias term
x_aug = np.column_stack([np.ones(len(x)),x_scaled])

In [ ]:
x_aug

In [12]:
# we need the gradient of the objective function
def gradient(w,x,y):
  error = y - x@w
  n = len(y)
  return 2/n*error@(-x)

In [13]:
w = [1,2,3,4]

In [14]:
gradient(w,x_aug,y)

array([-29.08039234, -34.23507179,  15.82110124,  13.94693778])

## Plain Gradient Descent

In [33]:
w = [1,2,3,4]
lr = 0.02
max_iter = 10000

In [24]:
# we need an iteration to update the weights:
for i in range(max_iter):
  w = w -lr*gradient(w,x_aug,y)

In [25]:
w

array([17.51295998,  3.54913919, 12.34623582,  0.40723538])

In [26]:
MSE(w,x_aug,y)

2.932288555695426

In [34]:
# you can also check the progress in updating the weights
# we need an iteration to update the weights:
for i in range(max_iter):
  w_new = w -lr*gradient(w,x_aug,y)
  if np.mean(abs(w_new-w))<1e-6:
    break
  w = w_new

In [35]:
w

array([18.55517774,  3.51263458, 14.46807804, -0.07908538])

In [36]:
i

8742

In [37]:
MSE(w,x_aug,y)

2.7841264823084977

## Mini-Batch Stochastic Gradient Descent

In [38]:
w = [1,2,3,4]
lr = 0.02
epochs = 10000
batch_size = 15

In [43]:
# the main idea is two nested for-loops
for i in range(epochs):
  for batch_ind in np.array_split(np.random.permutation(range(len(x))),batch_size):
    xs = x_aug[batch_ind]
    ys = y[batch_ind]
    w_new = w -lr*gradient(w,xs,ys)
    if np.mean(abs(w_new-w))<1e-6:
      break
    w = w_new

In [44]:
MSE(w,x_aug,y)

2.789447457075582

## RMSPROP

In [47]:
w = [1,2,3,4]
eta = 0.02
epsilon = 0.001
gamma =0.9
epochs = 10000
batch_size = 15
s = 0

In [48]:
# the main idea is two nested for-loops
for i in range(epochs):
  for batch_ind in np.array_split(np.random.permutation(range(len(x))),batch_size):
    xs = x_aug[batch_ind]
    ys = y[batch_ind]
    g = gradient(w,xs,ys)
    lr = eta/np.sqrt(s+epsilon)
    w_new = w -lr*g
    s = gamma*s+(1-gamma)*sum(g**2)
    if np.mean(abs(w_new-w))<1e-6:
      break
    w = w_new

In [49]:
MSE(w,x_aug,y)

2.7841531816171523